Imports

In [16]:
import pylangacq                    # reads CHAT-format transcripts from CHILDES
import pandas as pd                 # for building and saving our DataFrame
import os                           # for file path operations

Download the Brown Corpus directly from CHILDES

In [2]:
import pylangacq                                  # reads CHAT-format transcripts

# Path to the Brown folder you placed in data/raw/
folder_path = "../data/raw/Brown"                # local folder containing .cha files

# Parse all CHAT files found recursively inside the folder
chat = pylangacq.read_chat(folder_path)          # pylangacq scans all .cha files in subfolders

# Confirm how many files loaded — n_files is a property, not a method (no parentheses)
print(f"Number of files loaded: {chat.n_files}")  # expect ~300+


Number of files loaded: 214


The Brown corpus is Roger Brown's 1973 longitudinal study — the most cited study in child language development ever, and already in your proposal's references. It tracks 3 children only: Adam, Eve, and Sarah, recorded repeatedly over time as they grew up.
So 214 files = ~214 recording sessions spread across 3 children over several years of their lives. This is a longitudinal dataset (same children tracked over time), which is actually stronger for our research questions than many unrelated children recorded once.

| Our RQ | Why Brown corpus answers it |
|------|------|
| Do text representations recover developmental groupings? | 214 sessions with real age variation across years |
| Do groupings align with age? | Each session has exact age in months — perfect ground truth |
| Do lexical diversity measures differ across groups? | Longitudinal data shows *within-child* growth too |

The plan:

Now: Build and validate the entire pipeline on Brown (214 sessions). Get everything working cleanly.

Later if time permits: Add more CHILDES corpora (e.g. Eng-NA/MacWhinney, Eng-NA/Bloom) and re-run. 

In [5]:
# Inspect the Header object's available attributes and methods
header = chat.headers()[0]                        # get first session header

# dir() lists everything available on a Python object
print([x for x in dir(header) if not x.startswith('_')])  # filter out internal dunders

['comments', 'date', 'languages', 'location', 'media', 'number', 'options', 'other', 'participants', 'pid', 'recording_quality', 'room_layout', 'situation', 'tape_location', 'time_duration', 'time_start', 'transcriber', 'transcription', 'types', 'videos', 'warning']


In [6]:
# Inspect the participants field of the first session
header = chat.headers()[0]                        # get first session header

# participants likely contains a list — print it to see its structure
print("Participants:", header.participants)        # see all speaker info

# Also check the date field
print("Date:", header.date)                        # see how date is stored

Participants: [Participant(code='CHI', name='', role='Target_Child'), Participant(code='MOT', name='', role='Mother'), Participant(code='URS', name='', role='Investigator'), Participant(code='COL', name='', role='Investigator'), Participant(code='RIC', name='', role='Investigator')]
Date: 08-OCT-1962


In [10]:
# Check how ages are stored in pylangacq
ages = chat.ages()                                # get all ages across all sessions
print("First 5 ages:", ages[:5])                  # see the structure

First 5 ages: [Age('2;03.04'), Age('2;03.18'), Age('2;04.03'), Age('2;04.15'), Age('2;04.30')]


In [8]:
# file_paths is a property not a method — no parentheses needed
paths = chat.file_paths                           # list of all source file paths

# Print first 5 to see folder structure (will reveal Adam/Eve/Sarah)
print("First 5 file paths:")
for p in paths[:5]:                              # loop over first 5 paths
    print(p)                                     # print each path on its own line

First 5 file paths:
../data/raw/Brown/Adam/020304.cha
../data/raw/Brown/Adam/020318.cha
../data/raw/Brown/Adam/020403.cha
../data/raw/Brown/Adam/020415.cha
../data/raw/Brown/Adam/020430.cha


In [20]:
import os                                         # for extracting child name from file path

# --- PRE-COMPUTE per-file MLU and TTR using pylangacq's built-in methods ---
mlu_values = chat.mlu()                          # list of MLU floats, one per file, CHI only
ttr_values = chat.ttr()                          # list of TTR floats, one per file, CHI only

# --- GET UTTERANCES GROUPED BY FILE ---
all_utterances = chat.utterances(by_file=True)   # list of lists — one list per file

records = []                                      # empty list to collect one dict per session

for i in range(chat.n_files):                    # loop over every recording session

    # --- CHILD ID from file path ---
    path = chat.file_paths[i]                    # e.g. ../data/raw/Brown/Adam/020304.cha
    child_id = path.split('/')[-2]               # grab parent folder name = Adam, Eve, Sarah

    # --- AGE in months ---
    age = chat.ages()[i]                         # Age object e.g. Age('2;03.04')
    if age is None:                              # skip sessions with no recorded age
        continue
    age_months = age.years * 12 + age.months + age.days / 30  # convert to total months float

    # --- FILTER CHILD UTTERANCES FROM THIS SESSION ---
    session_utts = all_utterances[i]             # all utterances for this file
    child_utts = [u for u in session_utts        # keep only child utterances
                  if u.participant == 'CHI']     # filter by speaker code

    if len(child_utts) == 0:                     # skip sessions where child said nothing
        continue

    # --- BUILD DOCUMENT STRING ---
    words = []                                   # collect all child words this session
    for utt in child_utts:                       # loop over each child utterance
        tokens = [t.word.lower() for t in utt.tokens   # lowercase each token
                  if t.word and t.word.isalpha()]       # keep only alphabetic words
        words.extend(tokens)                     # add to session word list

    full_text = " ".join(words)                  # join all words into one document string

    # --- APPEND RECORD ---
    records.append({
        'file_id'      : i,                      # index of source file
        'child_id'     : child_id,               # Adam, Eve, or Sarah
        'age_months'   : round(age_months, 2),   # age in months as float
        'n_utterances' : len(child_utts),        # total child utterances this session
        'n_tokens'     : len(words),             # total filtered child words
        'mlu'          : mlu_values[i],          # mean length of utterance from pylangacq
        'ttr'          : ttr_values[i],          # type-token ratio from pylangacq
        'text'         : full_text,              # full child speech as one string
    })

print(f"Sessions extracted: {len(records)}")     # sanity check — expect ~200+

Sessions extracted: 214


In [21]:
df = pd.DataFrame(records)                       # convert list of dicts to DataFrame

print(f"Shape: {df.shape}")                      # (rows, columns)
print("\nData types:\n", df.dtypes)              # verify types look correct
print("\nFirst 5 rows:\n", df.head())            # eyeball the data
print("\nSummary stats:\n", df.describe())       # age range, MLU, TTR spread

Shape: (214, 8)

Data types:
 file_id           int64
child_id            str
age_months      float64
n_utterances      int64
n_tokens          int64
mlu             float64
ttr             float64
text                str
dtype: object

First 5 rows:
    file_id child_id  age_months  n_utterances  n_tokens   mlu       ttr  \
0        0     Adam       27.13          1268      2529  2.00  0.314286   
1        1     Adam       27.60          1284      2544  1.89  0.342857   
2        2     Adam       28.10           862      1919  2.33  0.351429   
3        3     Adam       28.50           783      1369  1.77  0.348571   
4        4     Adam       29.00           860      1779  1.93  0.374286   

                                                text  
0  play checkers big drum big drum big drum big d...  
1  okay my suitcase suitcase spaghetti monroe sui...  
2  yeah train there water water right there water...  
3  book read book up where book sitting chair dro...  
4  come cromer where t

In [22]:
# Check age range per child — expect Adam/Eve/Sarah to cover different ranges
print(df.groupby('child_id')['age_months'].agg(['min', 'max', 'count']))

# Check for suspiciously short sessions (fewer than 20 tokens)
short = df[df['n_tokens'] < 20]                  # filter very sparse sessions
print(f"\nVery short sessions (<20 tokens): {len(short)}")
print(short[['child_id', 'age_months', 'n_tokens']])  # inspect them

            min   max  count
child_id                    
Adam      27.13  62.4     55
Eve       18.00  27.0     20
Sarah     27.17  61.2    139

Very short sessions (<20 tokens): 0
Empty DataFrame
Columns: [child_id, age_months, n_tokens]
Index: []


In [23]:
output_path = "../data/processed/sessions.csv"   # destination path

df.to_csv(output_path, index=False)              # save without row index

print(f"Saved {len(df)} sessions → {output_path}")  # confirm

Saved 214 sessions → ../data/processed/sessions.csv


The Dataset at a Glance:

- Shape: (214, 8) → 214 sessions, 8 columns
- Age range: 18.0 → 62.4 months (1.5 years old → 5.2 years old)
- Zero short sessions → data quality is excellent, nothing to drop

Key insight:

- Eve covers the early developmental period, Adam and Sarah cover later development. 
- Together they span 18 to 62 months — that's the critical language acquisition window where the most dramatic growth happens.

MLU (Mean Length of Utterance) = average number of words per sentence a child produces. Roger Brown (1973) showed MLU is the single best predictor of language development, better than age alone.

MLU ~1: single words ("ball", "mama") → earliest stage
MLU ~2: two-word combinations ("big drum", "want cookie")
MLU ~3-4: simple sentences ("I want the red ball")
MLU ~5+: complex sentences with clauses

First 5 rows — Adam at 27 months has MLU=2.00, and by later sessions it climbs toward ~5. This is exactly the developmental growth our project is trying to discover with unsupervised methods.

TTR (Type-Token Ratio) = unique words ÷ total words. It measures vocabulary richness (lexical diversity — how varied the child's word choice is).

TTR=0.22 → very repetitive speech ("big drum big drum big drum" — literally in row 0's text)
TTR=0.56 → highly varied vocabulary
The mean of 0.38 is typical for child speech

One important caveat: TTR is sensitive to document length — longer texts naturally have lower TTR because words get repeated more. This is why we may want MTLD later (it corrects for length). But TTR is a fine starting point.